# Tennessee Data Center Interactive Map Generator

This notebook generates the interactive Tennessee data-center map from the deduplicated `Master` sheet in the data file.

The workflow is:

**Data file → cleaned records → JSON → HTML template → interactive map**

The page structure, styles, controls, and JavaScript are stored in a separate template file. Future interface changes should be made in `template.html` rather than in this notebook.

The template must contain the placeholder:

`__DATA_JSON__`

The notebook replaces that placeholder with the current facility records.

## 1. Configuration

Set the data path, template path, source sheet, state abbreviation, output path, and preview option here.

Required packages:
pandas, numpy, openpyxl, IPython

In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
from IPython.display import IFrame, display

DATA_FILE = Path("../dataset/tennessee_public_data_centers.xlsx")
TEMPLATE_FILE = Path("template/TN_dcmap_template.html")
DATA_SHEET = "Master"
STATE_ABBR = "TN"
OUTPUT_HTML = Path("tennessee_dcmap.html")
PREVIEW_IN_NOTEBOOK = True

## 2. JSON-safe value conversion

These helper functions normalize pandas and NumPy values before serialization and convert pipe-separated evidence URLs into a list.

In [2]:
def clean_value(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if math.isfinite(float(value)) else None
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")
    return str(value)


def parse_evidence_urls(value):
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    return [
        item.strip()
        for item in str(value).split("|")
        if item.strip().startswith(("http://", "https://"))
    ]

## 3. Load and normalize the data

The notebook uses the deduplicated `Master` sheet and keeps Tennessee records with valid coordinates.

The following fields are required:

`facility_id`, `facility_name`, `operator`, `state`, `latitude`, `longitude`, `facility_type`, `analysis_scope`, and `status_normalized`.

In [3]:
data = pd.read_excel(DATA_FILE, sheet_name=DATA_SHEET)

required_columns = {
    "facility_id",
    "facility_name",
    "operator",
    "state",
    "latitude",
    "longitude",
    "facility_type",
    "analysis_scope",
    "status_normalized",
}

missing = required_columns - set(data.columns)
if missing:
    raise KeyError(sorted(missing))

data = data[data["state"].astype(str).str.upper().eq(STATE_ABBR)].copy()
data["latitude"] = pd.to_numeric(data["latitude"], errors="coerce")
data["longitude"] = pd.to_numeric(data["longitude"], errors="coerce")

if "capacity_mw" in data.columns:
    data["capacity_mw"] = pd.to_numeric(data["capacity_mw"], errors="coerce")
else:
    data["capacity_mw"] = np.nan

data = data.dropna(subset=["latitude", "longitude"]).copy()

data["facility_type"].value_counts(dropna=False)

facility_type
data_center                 23
interconnection_facility    12
crypto_mining               10
Name: count, dtype: int64

## 4. Build the facility records

Each row is converted to the schema consumed by the map template. Optional values remain `null` when unavailable.

In [4]:
records = []

for _, row in data.iterrows():
    records.append(
        {
            "facility_id": clean_value(row.get("facility_id")),
            "facility_name": clean_value(row.get("facility_name")),
            "operator": clean_value(row.get("operator")) or "Unknown",
            "address": clean_value(row.get("address")),
            "city": clean_value(row.get("city")),
            "county": clean_value(row.get("county")),
            "state": clean_value(row.get("state")),
            "postal_code": clean_value(row.get("postal_code")),
            "lat": float(row["latitude"]),
            "lon": float(row["longitude"]),
            "location_precision": clean_value(row.get("location_precision")),
            "location_confidence": clean_value(row.get("location_confidence")),
            "status": clean_value(row.get("status_normalized")) or "unknown",
            "facility_type": clean_value(row.get("facility_type")) or "other",
            "analysis_scope": clean_value(row.get("analysis_scope")),
            "capacity_mw": None if pd.isna(row.get("capacity_mw")) else float(row["capacity_mw"]),
            "square_feet": clean_value(row.get("square_feet")),
            "property_acres": clean_value(row.get("property_acres")),
            "investment_usd": clean_value(row.get("investment_usd")),
            "sources": clean_value(row.get("sources")),
            "source_count": clean_value(row.get("source_count")),
            "last_updated": clean_value(row.get("last_updated")),
            "evidence_urls": parse_evidence_urls(row.get("evidence_urls")),
        }
    )

data_json = json.dumps(records, ensure_ascii=False, separators=(",", ":"))

len(records), sum(r["facility_type"] == "crypto_mining" for r in records)

(45, 10)

## 5. Load the HTML template

The complete page implementation is read from `TEMPLATE_FILE`. The notebook does not contain the HTML, CSS, or JavaScript source.

In [5]:
html_template = TEMPLATE_FILE.read_text(encoding="utf-8")

if "__DATA_JSON__" not in html_template:
    raise ValueError("__DATA_JSON__")

## 6. Generate the interactive map

The facility JSON replaces the template placeholder and the final HTML is written to `OUTPUT_HTML`.

In [6]:
final_html = html_template.replace("__DATA_JSON__", data_json, 1)
OUTPUT_HTML.write_text(final_html, encoding="utf-8")

53386

## 7. Preview

If `PREVIEW_IN_NOTEBOOK` is enabled, the generated HTML is displayed directly in the notebook.

In [7]:
if PREVIEW_IN_NOTEBOOK:
    display(IFrame(src=str(OUTPUT_HTML), width="100%", height=720))

## Updating the project

For data changes, update the `Master` sheet and rerun the notebook.

For interface, styling, basemap, filtering, popup, or JavaScript changes, edit `template.html` and rerun the notebook.

The notebook itself only handles data preparation, template loading, JSON injection, and HTML generation.